# Unit 01 · Data and the DRR

From a CT file on disk to a simulated X-ray pair (AP + LAT), plus the patient-level split.
The unit follows the videos; do each part right after its video:

| Part | After video | What you build | Suggested driver |
|---|---|---|---|
| A | 01 · What is an image | open a CT, the affine, the three planes, masks, count usable pelvises | Noa |
| B | 02 · Where the CT numbers come from | HU → attenuation, windowing | Adir |
| C | 03 · Cameras and projection | pinhole vs parallel projection | Adir |
| D | 04 · The DRR | a parallel-beam DRR, AP + LAT, then DiffDRR | Adir |
| E | – | the train/val/test split (fixed forever) | Noa |

Within a part, swap the keyboard on every TODO: one types, the other explains. Record it in `LOG.md`.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "01-data-and-drr"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
subjects = data.list_subjects()
print(len(subjects), "subjects on disk, e.g.", subjects[:5])
if data.HERO not in subjects and env.where() != "local":
    data.fetch_subjects(data.GOOD_SUBJECTS[:3])   # cloud: download 3 subjects (~150 MB) out of the remote zip

---
## Part A · What is an image (after video 01)

A CT file holds **a 3D array** (the voxels) and **an affine**: a 4×4 matrix that turns a voxel index
`(i, j, k)` into a position in millimetres `(x, y, z)` in the scanner's world.

In [ ]:
hu, affine = data.load_ct(data.HERO)
print("shape :", hu.shape, " dtype:", hu.dtype)
print("affine:\n", np.round(affine, 2))
print("axcodes:", data.axcodes(affine))

**Before you go on, answer together:** what do the three numbers of `shape` count? Which part of the affine
looks like a voxel size, and which part like a position? What does `('R', 'A', 'S')` tell you about index `k`?

### TODO A1 · `voxel_spacing`  ·  suggested driver: **Noa**

The voxel size in mm along each array axis, read **from the affine** (not from the header).
It has to work for any affine, including tilted ones whose 3×3 part isn't diagonal.

> **Nudge:** step one voxel along axis i: the index goes from (0,0,0) to (1,0,0). How far did the position in mm move, and which part of the affine says so?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def voxel_spacing(affine):
    """Voxel size (mm) along each array axis.

    affine  : (4, 4) voxel-index -> mm matrix
    returns : (3,) positive floats
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_voxel_spacing(voxel_spacing)

### TODO A2 · `voxel_to_mm`  ·  suggested driver: **Adir**

Turn voxel indices into world positions in mm with the affine. This is the first use of
**homogeneous coordinates** (you'll meet them again in video 03).

> **Nudge:** append a 1 to every (i, j, k). Why does that 1 make the translation column count, and how do you do it for N points at once?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def voxel_to_mm(affine, ijk):
    """World position (mm) of voxel indices.

    affine  : (4, 4)
    ijk     : (N, 3) voxel indices (int or float)
    returns : (N, 3) positions in mm
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_voxel_to_mm(voxel_to_mm)

In [ ]:
spacing = voxel_spacing(affine)
corners = np.array([[0, 0, 0], np.array(hu.shape) - 1])
print("voxel size (mm)        :", spacing)
print("first / last voxel (mm):\n", np.round(voxel_to_mm(affine, corners), 1))
print("scan size (cm)         :", np.round(np.array(hu.shape) * spacing / 10, 1))

Our CTs were resampled by TotalSegmentator to **1.5 mm** in all directions. Raw clinical CTs are often
anisotropic (e.g. 0.8 × 0.8 × 2.5 mm), which is why the checks use those numbers too.

Now the masks: one 0/1 file per bone, **on the same grid** as the CT.

In [ ]:
masks = data.load_mask(data.HERO, data.PELVIS + data.FEMURS)
for name, m in masks.items():
    print(f"{name:<12} {m.shape}  {m.sum():>8,} voxels")
pelvis = masks["hip_left"] | masks["hip_right"] | masks["sacrum"]   # the target of the whole project
centre = np.round(np.argwhere(pelvis).mean(axis=0)).astype(int)    # mean voxel index of the pelvis
print("pelvis centre voxel:", centre, "->", np.round(voxel_to_mm(affine, centre[None]), 1), "mm")

### TODO A3 · `orthogonal_planes`  ·  suggested driver: **Noa**

The three planes through a voxel: **axial** (cuts the head–feet axis), **coronal** (cuts the front–back
axis), **sagittal** (cuts the left–right axis). Use `codes` to find which array axis is which; don't
assume `('R', 'A', 'S')`, other datasets store volumes differently.

> **Nudge:** a plane is what's left when you fix one index. Which letter pair (R/L, A/P, S/I) belongs to each plane, and how do you find that pair's position in codes?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def orthogonal_planes(vol, ijk, codes):
    """The axial, coronal and sagittal slices through voxel ijk.

    vol     : (X, Y, Z) array
    ijk     : (i, j, k) voxel index the planes pass through
    codes   : axis codes such as ('R', 'A', 'S'), from data.axcodes(affine)
    returns : {"axial": 2D, "coronal": 2D, "sagittal": 2D}; keep the remaining two axes in array order
    """
    raise NotImplementedError("TODO A3")

In [ ]:
T.check_orthogonal_planes(orthogonal_planes)

In [ ]:
codes = data.axcodes(affine)
ct_planes = orthogonal_planes(np.clip(hu, -200, 1200), tuple(centre), codes)
mask_planes = orthogonal_planes(pelvis, tuple(centre), codes)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, name in zip(ax, ["axial", "coronal", "sagittal"]):
    a.imshow(ct_planes[name].T[::-1, ::-1], cmap="gray")          # display trick for RAS volumes, see below
    a.contour(mask_planes[name].T[::-1, ::-1].astype(float), levels=[0.5], colors="#ffb347", linewidths=1)
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

**Explain-back:** we display every plane with `p.T[::-1, ::-1]`. For an RAS volume, why does that put the
head at the top and the patient's right on the viewer's left (the radiology convention)? Would it still work
for an LPS volume? (In part D you'll write the general version.)

### TODO A4 · `mask_volume_cm3`  ·  suggested driver: **Adir**

The physical volume of a mask in cm³. A solid pelvis mask (both hip bones + sacrum) is roughly 1,000–1,500 cm³; the check tests that too.

> **Nudge:** every voxel is a small box of spacing[0] × spacing[1] × spacing[2] mm³. How many boxes are there, and how many mm³ in a cm³?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def mask_volume_cm3(mask, spacing_mm):
    """Volume of a boolean mask in cm^3. spacing_mm: (3,) voxel size in mm."""
    raise NotImplementedError("TODO A4")

In [ ]:
T.check_mask_volume_cm3(mask_volume_cm3)

In [ ]:
for name in data.PELVIS + data.FEMURS:
    print(f"{name:<12} {mask_volume_cm3(masks[name], spacing):7.1f} cm^3")
print(f"{'pelvis':<12} {mask_volume_cm3(pelvis, spacing):7.1f} cm^3")

### TODO A5 · `touches_scan_edge`  ·  suggested driver: **Noa**

Many CTs don't contain the whole pelvis: the scan's field of view stops in the middle of the bone.
A mask that reaches the first or last few slices along an axis was probably **cut**. We can't train on
those (the "ground truth" shape would be wrong).

> **Nudge:** find the lowest and highest index along `axis` where the mask has any voxel; compare them with the ends of the array.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def touches_scan_edge(mask, axis, margin=3):
    """True if the mask has a voxel within `margin` slices of either end along `axis`.

    Index n counts as the edge if n < margin or n >= mask.shape[axis] - margin.
    An empty mask touches nothing (False).
    """
    raise NotImplementedError("TODO A5")

In [ ]:
T.check_touches_scan_edge(touches_scan_edge)

Now the real question for the project: **how many subjects of the small subset have a complete pelvis?**
The loop is plumbing; the decision inside it is your function. Takes 1–2 minutes (it opens ~300 masks).

In [ ]:
import time
t0, counts, complete = time.time(), {"subjects": 0, "all 3 bones": 0, "complete pelvis": 0}, []
for sid in data.list_subjects():
    counts["subjects"] += 1
    m = data.load_mask(sid, data.PELVIS)
    if any(m[b].sum() <= 1000 for b in data.PELVIS):    # bone missing or a few stray voxels
        continue
    counts["all 3 bones"] += 1
    head_feet_axis = [c in "SI" for c in data.axcodes(data.load_nifti(data.ct_path(sid)).affine)].index(True)
    if not any(touches_scan_edge(m[b], head_feet_axis) for b in data.PELVIS):
        counts["complete pelvis"] += 1
        complete.append(sid)
print(counts, f"({time.time() - t0:.0f} s)")
screen = data.screening()
if screen:
    print("Claude's screening found", sum(r["complete"] for r in screen["results"].values()), "complete;",
          len(screen["good"]), "of them also have no metal near the pelvis")

**Explain-back:** what fraction of the subset is usable? If the full dataset (1,228 CTs) behaves the same,
about how many training pelvises do we get? Why is metal (hip prostheses, screws) a separate problem?
Write the numbers in `LOG.md`; the proposal's data estimate gets updated from them.

**Part A done.** Move `voxel_spacing`, `voxel_to_mm`, `orthogonal_planes`, `mask_volume_cm3` and
`touches_scan_edge` into `src/xray3d/volume.py`, then run `pytest` (cell at the end of the notebook).

---
## Part B · Where the CT numbers come from (after video 02)

A CT stores **Hounsfield units**: attenuation rescaled so water is 0 and air is −1000. A simulated X-ray
needs the attenuation coefficient **μ** itself (per mm), so the first step of every DRR is HU → μ.

In [ ]:
plotting.histogram(hu, bins=300, range=(-1300, 1700), title=f"HU values of {data.HERO}", xlabel="HU",
                   marks={"air": -1000, "fat": -100, "water": 0, "muscle": 40, "bone": 700})
print("min / max HU:", hu.min(), hu.max())

**Answer together:** find the air, fat and soft-tissue peaks. Where's bone, and why is it a long tail rather
than a peak? The minimum is below −1000: what could be "less than air"? (Hint: what's outside the scanner's
circular field of view?)

### TODO B1 · `hu_to_mu`  ·  suggested driver: **Adir**

Invert the Hounsfield definition from video 02, with μ_water = 0.02 mm⁻¹ (about right for diagnostic X-rays).
Attenuation can't be negative.

> **Nudge:** write HU = 1000·(μ − μ_w)/μ_w, then solve for μ. Then: what should happen to values below −1000?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def hu_to_mu(hu, mu_water=0.02):
    """Linear attenuation coefficient (1/mm) from Hounsfield units.

    hu       : array of HU (any shape, often int16)
    mu_water : attenuation of water in 1/mm
    returns  : float array, same shape, >= 0
    """
    raise NotImplementedError("TODO B1")

In [ ]:
T.check_hu_to_mu(hu_to_mu)

### TODO B2 · `window`  ·  suggested driver: **Noa**

**Windowing** maps a range of HU to display brightness 0..1: `center` is the middle grey, `width` the range
shown; everything below is black, above is white. Same CT, different windows show different tissues.

> **Nudge:** which HU becomes 0 and which becomes 1? Draw the line between those two points; then clip.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def window(hu, center, width):
    """Display brightness in [0, 1]: center -> 0.5, center -/+ width/2 -> 0 / 1, clipped outside."""
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_window(window)

In [ ]:
axial = orthogonal_planes(hu, tuple(centre), codes)["axial"].T[::-1, ::-1]
plotting.show_row(
    [window(axial, 40, 400), window(axial, 400, 1800), window(axial, -600, 1500), hu_to_mu(axial)],
    ["soft tissue (40 / 400)", "bone (400 / 1800)", "lung (-600 / 1500)", "mu (1/mm)"], size=4)

**Explain-back:** why does no single window show everything at once? In the μ image, what's the ratio between
bone and soft tissue? That ratio is what an X-ray "sees".

**Part B done.** Move `hu_to_mu` and `window` into `src/xray3d/physics.py`.

---
## Part C · Cameras and projection (after video 03)

An X-ray machine is a pinhole camera: the **source** is the pinhole, the **detector** is the image plane, and
the patient stands between them. Points are first moved into the camera's frame (`R`, `t`), then projected
with the intrinsics `K`.

### TODO C1 · `project_pinhole`  ·  suggested driver: **Adir**

Perspective projection with `K [R | t]`: camera coordinates `X_c = R X + t`, then `K`, then divide by depth.

> **Nudge:** after K you have (u·w, v·w, w) for every point. Which of the three is the depth, and what do you divide by it?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def project_pinhole(points, K, R, t):
    """Pixel coordinates of 3D points seen by a pinhole camera.

    points : (N, 3) world points (mm)
    K      : (3, 3) intrinsics (focal length and principal point, in pixels)
    R, t   : (3, 3) rotation and (3,) translation, world -> camera
    returns: (N, 2) pixel coordinates (u, v)
    """
    raise NotImplementedError("TODO C1")

In [ ]:
T.check_project_pinhole(project_pinhole)

### TODO C2 · `project_parallel`  ·  suggested driver: **Noa**

Parallel (orthographic) projection: rays don't meet at a point, they're all parallel. Depth is simply
dropped, and mm become pixels through the pixel size. Our first DRR (part D) works like this.

> **Nudge:** same R, t as before. Once you're in camera coordinates, which coordinate do you throw away, and how do mm become pixels?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def project_parallel(points, R, t, pixel_mm, principal):
    """Pixel coordinates under a parallel projection along the camera's z axis.

    points    : (N, 3) world points (mm)
    R, t      : world -> camera
    pixel_mm  : detector pixel size (mm)
    principal : (2,) pixel where the camera's z axis hits the detector
    returns   : (N, 2)
    """
    raise NotImplementedError("TODO C2")

In [ ]:
T.check_project_parallel(project_parallel)

Now both projections on the real pelvis. Claude placed a camera **in front of the patient** (an AP view):
the source is 850 mm anterior of the pelvis centre, looking backwards. Read the comments on `R`.

In [ ]:
pts = voxel_to_mm(affine, np.argwhere(pelvis)[::400])        # every 400th pelvis voxel, in mm
c_mm = pts.mean(axis=0)
# camera axes written in world (RAS) coordinates, one per row:
#   x_cam (image right)  = patient's Left   = -x_world
#   y_cam (image down)   = patient's feet   = -z_world
#   z_cam (viewing dir.) = front to back    = -y_world   (the beam goes A -> P)
R = np.array([[-1.0, 0, 0], [0, 0, -1], [0, -1, 0]])
source = c_mm + np.array([0, 850.0, 0])                      # 85 cm in front of the pelvis
t = -R @ source                                               # so that the source sits at the camera origin
K = np.array([[1020.0 / 1.5, 0, 160], [0, 1020.0 / 1.5, 160], [0, 0, 1]])   # detector 1,020 mm away, 1.5 mm pixels
uv_persp = project_pinhole(pts, K, R, t)
uv_par = project_parallel(pts, R, t, 1.5, np.array([160.0, 160.0]))
plotting.scatter2d([uv_par, uv_persp], ["parallel", "pinhole"], title="the pelvis, AP view")
w = lambda uv: uv[:, 0].max() - uv[:, 0].min()
print(f"width on the detector: parallel {w(uv_par) * 1.5:.0f} mm, pinhole {w(uv_persp) * 1.5:.0f} mm")

**Explain-back:** why is the pinhole image bigger? Compute the magnification by hand from the two distances
(source→pelvis 850 mm, source→detector 1,020 mm) and compare with the printed widths. Does the pelvis's
front or back get magnified more? Move the source to 500 mm and predict before you run.

**Part C done.** Move `project_pinhole` and `project_parallel` into `src/xray3d/geometry.py`.

---
## Part D · The DRR (after video 04)

A **digitally reconstructed radiograph**: send a ray through the μ volume for every detector pixel and add
up μ × step length. That sum is the line integral `p`; the detector measures `I = I0 · e^(−p)`.
We start with **parallel rays along an array axis**, so a ray is simply a line of voxels.

### TODO D1 · `parallel_drr`  ·  suggested driver: **Adir**

The line integral image for parallel rays along one array axis. Mind the units: μ is per mm, so every voxel
contributes μ × (its length along the ray in mm).

> **Nudge:** a ray along `axis` passes through every voxel of one line. How long is each voxel along that axis?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def parallel_drr(mu, spacing_mm, axis):
    """Line integral p = sum of mu * step along `axis` (parallel rays).

    mu         : (X, Y, Z) attenuation, 1/mm
    spacing_mm : (3,) voxel size in mm
    axis       : the array axis the rays travel along
    returns    : 2D array (the volume's shape without `axis`), unitless
    """
    raise NotImplementedError("TODO D1")

In [ ]:
T.check_parallel_drr(parallel_drr)

In [ ]:
mu = hu_to_mu(hu)
p = parallel_drr(mu, spacing, axis=1)               # axis 1 is A for this RAS volume: an AP view
plotting.show_row([p.T[::-1, ::-1], np.exp(-p).T[::-1, ::-1]],
                  ["line integral p", "detector intensity I / I0 = exp(-p)"], size=5)
print("p ranges from", round(float(p.min()), 2), "to", round(float(p.max()), 2))

**Answer together:** the right image is what the detector physically measures, yet the left one looks like the
X-rays you know. Why? (Look at the range of `p`: what is `e^(−7)`? Radiographs are displayed inverted and on a
log scale for exactly this reason.)

### TODO D2 · `ap_lat_views`  ·  suggested driver: **Adir**

Both views of any volume, **in the radiology orientation**, whatever the array layout:

* AP: rays along the front–back axis; rows run head → feet, columns run patient's Right → Left.
* LAT: rays along the left–right axis; rows run head → feet, columns run Anterior → Posterior.

This is the general version of the `.T[::-1, ::-1]` trick. The check tests RAS, LPS and a scrambled layout.

> **Nudge:** for each view: (1) find the beam axis in codes and call parallel_drr; (2) the image has two axes left; put the head–feet one first (transpose if needed); (3) flip each axis whose letter points the wrong way (e.g. 'S' means the index grows towards the head, but row 0 must be the head).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def ap_lat_views(mu, spacing_mm, codes):
    """AP and LAT line-integral images, oriented for display.

    returns (ap, lat):
      ap  : rows head->feet, columns patient's Right->Left
      lat : rows head->feet, columns Anterior->Posterior
    """
    raise NotImplementedError("TODO D2")

In [ ]:
T.check_ap_lat_views(ap_lat_views)

In [ ]:
ap, lat = ap_lat_views(mu, spacing, codes)
ap_bone, lat_bone = ap_lat_views(mu * pelvis, spacing, codes)   # only the pelvis voxels: what the network must recover
plotting.show_row([ap, lat, ap_bone, lat_bone], ["AP", "LAT", "AP, pelvis only", "LAT, pelvis only"], size=4)

**Explain-back:** in the LAT view, the two hip bones lie on top of each other. What does that mean for
the network, which has to separate them?

### D3 · A perspective DRR with DiffDRR (Claude's plumbing)

Real X-rays are perspective (part C), and DiffDRR renders them by tracing a ray from the source to every
detector pixel. It's also **differentiable**, which is used later in the project. It needs PyTorch, so run
this on **Colab** (or locally after `python -m pip install -e ".[drr]"`). Around 10–20 s on a CPU.

**Before you run it, predict:** the AP view uses rotation `(0, 0, 0)`. Which rotation turns the source
around the patient to give a lateral view?

In [ ]:
try:
    import torch
    from diffdrr.drr import DRR
    from diffdrr.data import read
except ImportError:
    print("DiffDRR isn't installed here. On Colab run:  !pip install -q diffdrr   then re-run this cell.")
else:
    import nibabel as nib
    # crop the CT to the pelvis (+3 cm), so the volume's centre is the pelvis's centre
    lo = np.maximum(np.argwhere(pelvis).min(axis=0) - 20, 0)
    hi = np.minimum(np.argwhere(pelvis).max(axis=0) + 20, pelvis.shape)
    crop = data.load_nifti(data.ct_path(data.HERO)).slicer[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    import tempfile   # a throwaway file DiffDRR reads once: keep it out of the data folder
    out = Path(tempfile.gettempdir()) / f"{data.HERO}_pelvis_crop.nii.gz"
    nib.save(crop, out)

    subject = read(str(out), orientation="AP")
    drr = DRR(subject, sdd=1020.0, height=256, delx=1.5)    # source-detector 1,020 mm, 256 x 256 pixels of 1.5 mm
    views = {"AP": [0.0, 0.0, 0.0], "LAT": [torch.pi / 2, 0.0, 0.0]}   # euler angles (Z, X, Y) in radians
    images = []
    for name, rot in views.items():
        img = drr(torch.tensor([rot]), torch.tensor([[0.0, 850.0, 0.0]]),   # source 850 mm from the volume centre
                  parameterization="euler_angles", convention="ZXY")
        images.append(img[0, 0].cpu().numpy())
    ap_crop, _ = ap_lat_views(mu[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]], spacing, codes)   # yours, same crop
    plotting.show_row(images + [ap_crop], ["DiffDRR AP (perspective)", "DiffDRR LAT", "your parallel AP, same crop"], size=4.5)

**Explain-back:** compare your parallel AP with DiffDRR's AP of the same crop. What differs, and which parts of part C
explain it? (The dark frame around DiffDRR's images is the edge of the cropped volume: outside it there's nothing to attenuate.)

**Part D done.** Move `parallel_drr` and `ap_lat_views` into `src/xray3d/drr.py`.

---
## Part E · The split, fixed forever

Train / validation / test = 70 / 10 / 20 **by subject**. The split is made once, over all 1,228 subject IDs
of the full dataset (`splits/subjects_v201.txt`), and saved to `splits/split_v1.json`. Every later unit reads
that file. It's **never regenerated**: if it changes, earlier results can no longer be compared.

### TODO E1 · `split_subjects`  ·  suggested driver: **Noa**

A deterministic, patient-level split. The same IDs must land in the same part every time, **even if the list
arrives in a different order**.

> **Nudge:** what does a random shuffle need to give the same answer twice, and what must you do to the list first so the order it came in doesn't matter?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def split_subjects(ids, fractions=(0.7, 0.1, 0.2), seed=0):
    """Split subject ids into {"train": [...], "val": [...], "test": [...]} (no subject in two parts)."""
    raise NotImplementedError("TODO E1")

In [ ]:
T.check_split_subjects(split_subjects)

In [ ]:
import json, datetime
ids = data.all_subject_ids()
mine = split_subjects(ids)
path = env.repo_root() / "splits" / "split_v1.json"
if path.exists():
    saved = json.loads(path.read_text())
    same = all(sorted(saved[k]) == sorted(mine[k]) for k in ("train", "val", "test"))
    print("split_v1.json already exists; NOT overwriting.", "Your function reproduces it." if same else
          "WARNING: your function gives a different split. Use the file, not the function.")
else:
    path.write_text(json.dumps({"version": 1, "seed": 0, "fractions": [0.7, 0.1, 0.2],
                                "created": str(datetime.date.today()), "source": "splits/subjects_v201.txt",
                                **{k: sorted(v) for k, v in mine.items()}}, indent=1))
    print("wrote", path)
split = json.loads(path.read_text())
print({k: len(split[k]) for k in ("train", "val", "test")})
print("our 27 usable small-subset pelvises:", {k: sum(s in split[k] for s in data.GOOD_SUBJECTS) for k in ("train", "val", "test")})

**Explain-back:** why split by subject rather than by image (each CT gives many DRRs)? Why split *all* 1,228
IDs now, before we know which are usable? What would go wrong if we re-ran the split with a new seed in week 5?

**Part E done.** Move `split_subjects` into `src/xray3d/split.py`.

---
## Wrap-up: run the checks against `src/xray3d`

Functions you haven't moved yet show up as **skipped**, moved ones as **passed**.

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md`: driver per TODO, hints used, the explain-back answers and the numbers (usable pelvises,
the split sizes). Then ask Claude to review the code like a TA, and to commit.